# Guided contribution: a deterministic financial transition fixture

This notebook is an onboarding exercise for contributing a small, testable financial time-series fixture to FeatureGraph.

The goal is **not** to build a trading strategy or show that any interval predicts returns. The goal is to construct an explicit representation:

1. ordered observations,
2. sample-level states and boundary events,
3. one row per temporally bounded interval,
4. exact validation checks.

By the end, you should be able to explain every boundary in the interval table and convert the checks into a focused pull request.


## Contribution contract

Keep these constraints fixed throughout the exercise:

- Use no network calls, broker APIs, credentials, or downloaded market data.
- Use the supplied values exactly; do not tune them to produce a preferred result.
- Treat `rising`, `falling`, and `inactive` as analytical states, not market interpretations.
- Preserve the raw `price` column. Derived columns must be added separately.
- Keep the first sample explicitly unclassified because it has no preceding observation.
- Preserve incomplete intervals rather than silently treating them as complete.
- Make only structural and analytical claims. Scientific or financial interpretation is outside this notebook's scope.


## 0. Environment

Run this notebook from a clone of `featuregraph/featuregraph`. From the repository root, install the development environment once:

```bash
python -m pip install -e ".[dev,notebooks]"
```

Then start JupyterLab from the same environment. The import check below intentionally fails with a useful message if the repository package is unavailable.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import featuregraph as fg
    from featuregraph.operators.events import enter_label, exit_label
    from featuregraph.operators.states import (
        falling_state,
        inactive_state,
        negative_state,
        positive_state,
        rising_state,
        stable_state,
    )
except ImportError as exc:
    raise ImportError(
        "Install FeatureGraph from the repository root with "
        "`python -m pip install -e '.[dev,notebooks]'`."
    ) from exc

pd.set_option("display.max_columns", 30)


## 1. Declare the observation fixture

The fixture is deliberately small and fully explicit. It is **price-like**, not a simulation of a real market. Its purpose is to make every expected state and boundary knowable before the implementation runs.

Before executing the next cell, predict where the series is rising, falling, and unchanged. Write the predicted sample ranges in your notes.


In [ ]:
price = [
    100.0,
    100.0,
    101.0,
    102.0,
    103.0,
    103.0,
    103.0,
    102.0,
    101.0,
    99.0,
    99.0,
    99.0,
    100.0,
    101.0,
    101.0,
    101.0,
]

observations = pd.DataFrame(
    {
        "sample": np.arange(len(price), dtype=int),
        "timestamp": pd.date_range("2026-01-01", periods=len(price), freq="D"),
        "price": price,
    }
)

observations


In [ ]:
assert observations["sample"].is_monotonic_increasing
assert observations["timestamp"].is_monotonic_increasing
assert observations["timestamp"].is_unique
assert observations["price"].notna().all()
assert len(observations) == 16


In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(observations["sample"], observations["price"], marker="o")
ax.set(
    title="Deterministic price-like observation fixture",
    xlabel="Sample",
    ylabel="Price-like value",
)
ax.grid(alpha=0.25)
plt.show()


## 2. Write the state contract before constructing states

For sample $t > 0$, define

$$\Delta_t = price_t - price_{t-1}.$$

With a fixed threshold $\varepsilon = 0$:

| State | Predicate | Meaning in this notebook |
|---|---|---|
| `rising` | $\Delta_t > \varepsilon$ | the value increased from the preceding sample |
| `falling` | $\Delta_t < -\varepsilon$ | the value decreased from the preceding sample |
| `inactive` | $|\Delta_t| \leq \varepsilon$ | the value did not change beyond the threshold |
| `unclassified` | $t=0$ | no preceding observation exists |

One interval is a maximal consecutive run of one label. Its start and end are determined by label changes, not by a fixed rolling window.

Pause before continuing: Why must preprocessing or smoothing be declared here rather than added after looking at the interval table? This fixture uses no smoothing.


In [ ]:
EPSILON = 0.0

observations["delta"] = observations["price"].diff()
observations["rising"] = positive_state(observations["delta"], eps=EPSILON)
observations["falling"] = negative_state(observations["delta"], eps=EPSILON)
observations["inactive"] = inactive_state(observations["delta"], eps=EPSILON)

observations[["sample", "price", "delta", "rising", "falling", "inactive"]]


### Exclusivity check

Every sample after the first must belong to exactly one primitive state. The first sample must belong to none of them. This is a representation invariant, not a visual preference.


In [ ]:
state_membership_count = observations[["rising", "falling", "inactive"]].sum(axis=1)

assert state_membership_count.iloc[0] == 0
assert state_membership_count.iloc[1:].eq(1).all()


In [ ]:
observations["state"] = np.select(
    [
        observations["rising"],
        observations["falling"],
        observations["inactive"],
    ],
    ["rising", "falling", "inactive"],
    default="unclassified",
)

EXPECTED_STATES = [
    "unclassified",
    "inactive",
    "rising",
    "rising",
    "rising",
    "inactive",
    "inactive",
    "falling",
    "falling",
    "falling",
    "inactive",
    "inactive",
    "rising",
    "rising",
    "inactive",
    "inactive",
]

assert observations["state"].tolist() == EXPECTED_STATES
observations[["sample", "price", "delta", "state"]]


## 3. Detect observed entry and exit events

`enter_label` identifies label changes. `exit_label` looks forward to determine whether the current label ends at this sample.

We use `include_first=False` because sample 0 has no observed entry boundary. We use `include_last=False` because reaching the end of the dataset is not evidence that the final state actually ended. This keeps boundary-truncated intervals visible.


In [ ]:
observations["enter_state"] = enter_label(
    observations["state"], include_first=False
)
observations["exit_state"] = exit_label(
    observations["state"], include_last=False
)

# A separate partition marker includes the first row so every sample receives an ID.
observations["start_partition"] = enter_label(
    observations["state"], include_first=True
)
observations["interval_id"] = observations["start_partition"].cumsum() - 1

observations[
    ["sample", "price", "state", "enter_state", "exit_state", "interval_id"]
]


In [ ]:
EXPECTED_ENTER_SAMPLES = [1, 2, 5, 7, 10, 12, 14]
EXPECTED_EXIT_SAMPLES = [0, 1, 4, 6, 9, 11, 13]

assert observations.loc[observations["enter_state"], "sample"].tolist() == EXPECTED_ENTER_SAMPLES
assert observations.loc[observations["exit_state"], "sample"].tolist() == EXPECTED_EXIT_SAMPLES
assert observations["interval_id"].tolist() == [0, 1, 2, 2, 2, 3, 3, 4, 4, 4, 5, 5, 6, 6, 7, 7]


## 4. Construct one row per interval

The first observed sample in a rising or falling state is already one step beyond its preceding value. Therefore, `end_value - first_value` omits the first change.

The object table retains both measurements:

- `within_run_change`: end value minus the first sample carrying the label.
- `boundary_net_change`: end value minus the value immediately before the label began.

For transition magnitude, the boundary-aware quantity is the relevant one. This distinction should be explicit rather than hidden in an aggregation.


In [ ]:
intervals = (
    observations.groupby(["interval_id", "state"], sort=False)
    .agg(
        start_sample=("sample", "first"),
        end_sample=("sample", "last"),
        start_time=("timestamp", "first"),
        end_time=("timestamp", "last"),
        state_sample_count=("sample", "size"),
        first_labeled_value=("price", "first"),
        end_value=("price", "last"),
    )
    .reset_index()
)

preceding_sample = intervals["start_sample"] - 1
preceding_value = preceding_sample.map(observations.set_index("sample")["price"])

intervals["preceding_sample"] = preceding_sample.where(preceding_sample.ge(0))
intervals["preceding_value"] = preceding_value
intervals["duration_steps"] = intervals["state_sample_count"]
intervals["within_run_change"] = (
    intervals["end_value"] - intervals["first_labeled_value"]
)
intervals["boundary_net_change"] = intervals["end_value"] - intervals["preceding_value"]
intervals["has_start_boundary"] = intervals["preceding_sample"].notna()
intervals["has_end_boundary"] = intervals["end_sample"].lt(observations["sample"].max())

# Sample 0 is necessary observation context, but it is not a constructed transition interval.
transition_intervals = intervals.loc[
    intervals["state"].ne("unclassified")
].reset_index(drop=True)

transition_intervals


### Exact interval validation

These expected values were derived from the declared contract before treating the output as correct. Do not weaken an assertion merely because an implementation produces a different result; first determine whether the contract, implementation, or expectation is wrong.


In [ ]:
expected_intervals = pd.DataFrame(
    {
        "state": [
            "inactive",
            "rising",
            "inactive",
            "falling",
            "inactive",
            "rising",
            "inactive",
        ],
        "start_sample": [1, 2, 5, 7, 10, 12, 14],
        "end_sample": [1, 4, 6, 9, 11, 13, 15],
        "duration_steps": [1, 3, 2, 3, 2, 2, 2],
        "boundary_net_change": [0.0, 3.0, 0.0, -4.0, 0.0, 2.0, 0.0],
        "has_start_boundary": [True] * 7,
        "has_end_boundary": [True, True, True, True, True, True, False],
    }
)

pd.testing.assert_frame_equal(
    transition_intervals[expected_intervals.columns].reset_index(drop=True),
    expected_intervals,
    check_dtype=False,
)

assert transition_intervals["interval_id"].is_unique
assert transition_intervals.iloc[-1]["has_end_boundary"] == False  # noqa: E712


In [ ]:
state_colors = {
    "rising": "#2a9d8f",
    "falling": "#e76f51",
    "inactive": "#8d99ae",
}

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(observations["sample"], observations["price"], color="black", alpha=0.45)

for _, interval in transition_intervals.iterrows():
    mask = observations["interval_id"].eq(interval["interval_id"])
    ax.plot(
        observations.loc[mask, "sample"],
        observations.loc[mask, "price"],
        marker="o",
        linewidth=3,
        color=state_colors[interval["state"]],
        label=interval["state"],
    )

handles, labels = ax.get_legend_handles_labels()
unique = dict(zip(labels, handles))
ax.legend(unique.values(), unique.keys(), title="Analytical state")
ax.set(xlabel="Sample", ylabel="Price-like value", title="Explicit interval representation")
ax.grid(alpha=0.2)
plt.show()


## 5. Verify the current `Transition` API one direction at a time

The complete categorical interval table above represents all three states together. The current `Transition` class materializes one predicate at a time. Use a fresh copy for each direction because the class writes derived columns into the supplied DataFrame.

This section verifies current behavior; it does not claim that the current summary API already produces the categorical interval table above.


In [ ]:
direction_cases = {
    "rising": {
        "operator": rising_state,
        "expected_true_samples": [2, 3, 4, 12, 13],
        "expected_enter_samples": [2, 12],
        "expected_exit_samples": [5, 14],
    },
    "falling": {
        "operator": falling_state,
        "expected_true_samples": [7, 8, 9],
        "expected_enter_samples": [7],
        "expected_exit_samples": [10],
    },
    "inactive": {
        "operator": stable_state,
        "expected_true_samples": [1, 5, 6, 10, 11, 14, 15],
        "expected_enter_samples": [1, 5, 10, 14],
        "expected_exit_samples": [2, 7, 12],
    },
}

materialized = {}

for direction, case in direction_cases.items():
    frame = observations[["sample", "timestamp", "price"]].copy()
    behavior = fg.transition.Transition(
        frame,
        signal="price",
        direction=direction,
        op=case["operator"],
        eps=EPSILON,
    )
    materialized[direction] = behavior.df

    state_col = f"price_{direction}"
    enter_col = f"enter_price_{direction}"
    exit_col = f"exit_price_{direction}"

    assert frame.loc[frame[state_col], "sample"].tolist() == case["expected_true_samples"]
    assert frame.loc[frame[enter_col], "sample"].tolist() == case["expected_enter_samples"]
    assert frame.loc[frame[exit_col], "sample"].tolist() == case["expected_exit_samples"]

materialized["rising"]


## 6. Threshold sensitivity is part of the contract

A tiny floating-point residue can look like a change when $\varepsilon=0$. The threshold is therefore part of the representation specification. It must be declared before evaluating the objects it produces.

Predict the state at each sample below for both thresholds before running the cell.


In [ ]:
chatter = pd.Series([100.0, 100.0 + 1e-13, 100.0, 100.01], name="price")

threshold_comparison = pd.DataFrame(
    {
        "price": chatter,
        "delta": chatter.diff(),
        "rising_eps_0": rising_state(chatter, eps=0.0),
        "rising_eps_1e_12": rising_state(chatter, eps=1e-12),
        "stable_eps_1e_12": stable_state(chatter, eps=1e-12),
    }
)

assert threshold_comparison["rising_eps_0"].tolist() == [False, True, False, True]
assert threshold_comparison["rising_eps_1e_12"].tolist() == [False, False, False, True]
threshold_comparison


## 7. Turn the exercise into a small PR

The first contribution should remain narrow. A good PR can contain this notebook plus fixture-focused tests, without changing the compiler or redesigning `Transition`.

Suggested test coverage:

1. The supplied values create the exact rising, falling, and inactive sample sets.
2. Entry and exit samples match the declared contract.
3. Interval IDs and boundaries match the expected object table.
4. The final inactive interval is retained with `has_end_boundary=False`.
5. A nonzero epsilon rejects floating-point chatter.
6. No test depends on the network or current market data.

If current library behavior prevents one of these expectations, document the mismatch before proposing a core change. Do not revise the expected result simply to match the implementation.


In [ ]:
# Compact fixture values suitable for extraction into a pytest fixture.
FINANCIAL_TRANSITION_PRICES = tuple(price)

# Exact categorical expectations suitable for parameterized tests.
EXPECTED_TRUE_SAMPLES = {
    "rising": (2, 3, 4, 12, 13),
    "falling": (7, 8, 9),
    "inactive": (1, 5, 6, 10, 11, 14, 15),
}

assert len(FINANCIAL_TRANSITION_PRICES) == 16
assert set(EXPECTED_TRUE_SAMPLES) == {"rising", "falling", "inactive"}


## 8. Contributor reflection

Before opening the PR, answer these questions in the PR description:

1. What makes an interval begin and end?
2. Why is sample 0 unclassified?
3. Why does `boundary_net_change` differ from `within_run_change`?
4. Why is the final inactive interval boundary-truncated?
5. Which choices are structural, which are analytical, and which would require financial interpretation?
6. What would have to be declared before applying this construction to a real financial series?

### Completion checklist

- [ ] Restart the kernel and run all cells from top to bottom.
- [ ] Confirm that every assertion passes.
- [ ] Confirm that no downloaded data, API key, or machine-specific path was added.
- [ ] Keep generated tables inspectable in the notebook.
- [ ] Describe the state contract and threshold in the PR.
- [ ] Report limitations without making predictive or trading claims.

The successful outcome is a deterministic, reviewable representation—not an attractive financial result.
